# Swiss Legal Citation Retrieval — Kaggle Submission

**Offline pipeline** (no Claude API required for Kaggle GPU environment).

Steps:
1. Install dependencies
2. Build BM25 + Dense (multilingual-e5-large) indices
3. Hybrid retrieval with RRF fusion
4. Threshold-based filtering → submission.csv

> For re-ranking with Claude API, set `USE_RERANKER = True` and provide `ANTHROPIC_API_KEY`.

In [ ]:
# ── Configuration ────────────────────────────────────────────────────────────
USE_RERANKER   = False          # Set True + provide API key to use Claude reranking
ANTHROPIC_KEY  = ""             # or use os.environ['ANTHROPIC_API_KEY']

BM25_TOP_K     = 100
DENSE_TOP_K    = 100
RRF_K          = 60
RERANK_TOP_K   = 20
THRESHOLD      = 0.5
DENSE_MODEL    = "intfloat/multilingual-e5-large"
RERANK_MODEL   = "claude-sonnet-4-20250514"
SEP            = ";"

import os, sys
from pathlib import Path

# Kaggle environment paths
KAGGLE_INPUT = Path("/kaggle/input")
WORKING_DIR  = Path("/kaggle/working")
REPO_DIR     = Path("/kaggle/input/omnilex-starter")   # adjust to dataset slug

# Add omnilex library to path
sys.path.insert(0, str(REPO_DIR / "src"))

print("Config ready")

In [ ]:
# ── Install dependencies ──────────────────────────────────────────────────────
!pip install -q rank-bm25 sentence-transformers faiss-cpu python-dotenv pyyaml
if USE_RERANKER:
    !pip install -q anthropic

In [ ]:
# ── Load corpus ───────────────────────────────────────────────────────────────
import json
import pandas as pd

DATA_DIR = KAGGLE_INPUT / "llm-agentic-legal-information-retrieval"

from omnilex.retrieval.bm25_index import load_jsonl_corpus

laws   = load_jsonl_corpus(str(DATA_DIR / "federal_laws.jsonl"))
courts = load_jsonl_corpus(str(DATA_DIR / "court_decisions.jsonl"))
test_df = pd.read_csv(DATA_DIR / "test.csv")

print(f"Laws: {len(laws)}, Courts: {len(courts)}, Queries: {len(test_df)}")

In [ ]:
# ── Build BM25 indices ────────────────────────────────────────────────────────
from omnilex.retrieval.bm25_index import BM25Index

bm25_laws   = BM25Index(text_field="text", citation_field="citation")
bm25_courts = BM25Index(text_field="text", citation_field="citation")

bm25_laws.build(laws)
bm25_courts.build(courts)
print("BM25 indices built")

In [ ]:
# ── Build dense (FAISS) indices ───────────────────────────────────────────────
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(DENSE_MODEL)
PREFIX = "passage: "

def encode_corpus(docs):
    texts = [PREFIX + (d.get("text") or d.get("regeste") or "") for d in docs]
    embs  = model.encode(texts, batch_size=64, normalize_embeddings=True,
                         show_progress_bar=True)
    return embs.astype("float32")

laws_embs   = encode_corpus(laws)
courts_embs = encode_corpus(courts)

dim = laws_embs.shape[1]
faiss_laws   = faiss.IndexFlatIP(dim); faiss_laws.add(laws_embs)
faiss_courts = faiss.IndexFlatIP(dim); faiss_courts.add(courts_embs)
print(f"FAISS indices: laws={faiss_laws.ntotal}, courts={faiss_courts.ntotal}")

In [ ]:
# ── Hybrid retrieval + RRF ────────────────────────────────────────────────────
from collections import defaultdict

def bm25_search(index, query, top_k):
    scores = index.bm25.get_scores(index.tokenize(query))
    ranked = sorted(zip(index.documents, scores), key=lambda x: x[1], reverse=True)
    return ranked[:top_k]

def dense_search(faiss_idx, meta, query, top_k):
    q = model.encode(["query: " + query], normalize_embeddings=True).astype("float32")
    scores, idxs = faiss_idx.search(q, top_k)
    return [(meta[i], float(s)) for s, i in zip(scores[0], idxs[0]) if i >= 0]

def rrf(ranked_lists, weights, k=60):
    scores = defaultdict(float)
    for rl, w in zip(ranked_lists, weights):
        for rank, (cid, _) in enumerate(rl, 1):
            scores[cid] += w / (rank + k)
    return scores

candidates_all = []
for _, row in test_df.iterrows():
    q = row["query"]

    bl = bm25_search(bm25_laws,   q, BM25_TOP_K)
    bc = bm25_search(bm25_courts, q, BM25_TOP_K)
    dl = dense_search(faiss_laws,   laws,   q, DENSE_TOP_K)
    dc = dense_search(faiss_courts, courts, q, DENSE_TOP_K)

    all_docs = {}
    for doc, _ in bl + bc + dl + dc:
        cid = doc.get("citation") or doc.get("id") or ""
        all_docs[cid] = doc

    bm25_ranked  = [(d.get("citation",""), s) for d,s in sorted(bl+bc, key=lambda x:-x[1])]
    dense_ranked = [(d.get("citation",""), s) for d,s in sorted(dl+dc, key=lambda x:-x[1])]

    rrf_scores = rrf([bm25_ranked, dense_ranked], [1.0, 1.0], k=RRF_K)
    sorted_cids = sorted(rrf_scores, key=rrf_scores.get, reverse=True)

    candidates_all.append({
        "query_id": row["query_id"],
        "query": q,
        "candidates": [{"citation": c, "rrf_score": rrf_scores[c]} for c in sorted_cids],
    })

print(f"Retrieved candidates for {len(candidates_all)} queries")

In [ ]:
# ── (Optional) Claude reranking ───────────────────────────────────────────────
if USE_RERANKER:
    import anthropic, time

    client = anthropic.Anthropic(api_key=ANTHROPIC_KEY or os.environ.get("ANTHROPIC_API_KEY"))

    RERANK_PROMPT = """You are an expert in Swiss law. Given a legal query and candidates, \
score each citation 0.0-1.0 for relevance.\n\nQuery: {query}\n\nCandidates:\n{block}\n\n\
Return JSON array: [{{'citation': '<cit>', 'score': <float>}}, ...]\nOnly JSON, no other text."""

    def rerank(query, candidates):
        block = "\n".join(f"{i+1}. [{c['citation']}]" for i,c in enumerate(candidates))
        resp  = client.messages.create(
            model=RERANK_MODEL, max_tokens=1024,
            messages=[{"role":"user","content": RERANK_PROMPT.format(query=query,block=block)}]
        )
        raw = resp.content[0].text.strip()
        if raw.startswith("```"): raw = raw.split("```")[1].lstrip("json").strip()
        scored = json.loads(raw)
        return {item["citation"]: float(item["score"]) for item in scored}

    for rec in candidates_all:
        top = rec["candidates"][:RERANK_TOP_K]
        try:
            scores = rerank(rec["query"], top)
            for c in rec["candidates"]:
                c["rerank_score"] = scores.get(c["citation"], 0.0)
            time.sleep(0.2)
        except Exception as e:
            print(f"  Rerank failed for {rec['query_id']}: {e}")
            for c in rec["candidates"]:
                c["rerank_score"] = c["rrf_score"]

    print("Reranking done")
else:
    # Use RRF score directly
    for rec in candidates_all:
        for c in rec["candidates"]:
            c["rerank_score"] = c["rrf_score"]
    print("Skipped reranking — using RRF scores")

In [ ]:
# ── Generate submission.csv ───────────────────────────────────────────────────
from omnilex.citations.normalizer import CitationNormalizer

normalizer = CitationNormalizer()

# Normalize scores to [0,1] for threshold comparison when using RRF
# (RRF scores are small positives; we use rank-based cutoff instead)
TOP_N = RERANK_TOP_K  # keep at most this many per query when no reranker

rows = []
for rec in candidates_all:
    candidates = sorted(rec["candidates"], key=lambda x: x["rerank_score"], reverse=True)

    if USE_RERANKER:
        kept = [c["citation"] for c in candidates if c["rerank_score"] >= THRESHOLD]
    else:
        kept = [c["citation"] for c in candidates[:TOP_N]]

    # Canonicalize
    canonical = []
    seen = set()
    for raw in kept:
        cid = normalizer.canonicalize(raw)
        if cid and cid not in seen:
            seen.add(cid)
            canonical.append(cid)

    rows.append({"query_id": rec["query_id"], "predicted_citations": SEP.join(canonical)})

submission_df = pd.DataFrame(rows)
out_path = WORKING_DIR / "submission.csv"
submission_df.to_csv(out_path, index=False)

print(f"Saved {len(submission_df)} rows → {out_path}")
submission_df.head()

In [ ]:
# ── Validate submission format ─────────────────────────────────────────────────
from omnilex.evaluation.scorer import validate_submission_format

errors = validate_submission_format(str(out_path))
if errors:
    print("Validation FAILED:")
    for e in errors: print(f"  - {e}")
else:
    print("Submission format: VALID ✓")

print(f"\nSummary:")
print(f"  Total queries: {len(submission_df)}")
empty = (submission_df['predicted_citations'] == '').sum()
print(f"  Empty predictions: {empty}")
avg_cits = submission_df['predicted_citations'].apply(
    lambda x: len(x.split(SEP)) if x else 0
).mean()
print(f"  Avg citations/query: {avg_cits:.1f}")